In [1]:
from dotenv import load_dotenv
from agents import Agent,Runner,trace ,function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail ,Email , To, Content

In [2]:
load_dotenv(override=True)

True

In [4]:
#step1:agent workflow

instruction1 = "you are a sale agent working for abcAI.\
    a company provide check user satisfcation for sales items SOC 2 compliance and SOC 2 audits. you worte professional,serious cold emails"


instruction2="you are a hunorous,engaging sales agent working for abcAI.\
      a company provide check user satisfcation for sales items SOC 2 compliance and SOC 2 audits. you worte FUNNY, cold emails ARE LIKELY TO GT A RESPONSE "

instruction3="you are a busy sales agent working in abcAI.\
    A COMPANY PROVIDE CHECK USER SATISFICATION FOR SALES ITEMS SOC 2 compliance and SOC 2 audits.,write concise,to the point cold emails "

In [5]:
sales_agent1 = Agent(
    name="Professional Sale Agent",
    instructions = instruction1,
    model="gpt-4o-mini"
)
sales_agent2 = Agent(
    name="Professional Sale Agent",
    instructions = instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Professional Sale Agent",
    instructions = instruction3,
    model="gpt-4o-mini"
)

In [6]:
result = Runner.run_streamed(sales_agent1,input="write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data,ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

OPENAI_API_KEY is not set, skipping trace export


In [7]:
message="write a cold sales email"

with trace("parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message)
    )



outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

NameError: name 'asyncio' is not defined

OPENAI_API_KEY is not set, skipping trace export


In [17]:
##SELECT BEST EMAIL PICKER

sales_picker =Agent(
    name="sales picker",
    instructions="you pick the best email from the given options.\
        select best email from thee 3 option"
)



In [ ]:
message="write a sale email"
with trace(name="slectionf frm salae people"):
    results=await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message),
    )

    outputs = [result.final_output for result in results]
    best = await Runner.run(sales_picker,output)
    print(f"best sales email:\n {best.final_outputs}")

    1


In [18]:
###tools creation

sales_agent1 = Agent(
    name="Professional Sale Agent",
    instructions = instruction1,
    model="gpt-4o-mini"
)
sales_agent2 = Agent(
    name="Professional Sale Agent",
    instructions = instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Professional Sale Agent",
    instructions = instruction3,
    model="gpt-4o-mini"
)

In [19]:
sales_agent1

Agent(name='Professional Sale Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='you are a sale agent working for abcAI.    a company provide check user satisfcation for sales items SOC 2 compliance and SOC 2 audits. you worte professional,serious cold emails', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [20]:
##tool and agent interactions

@function_tool
def send_email(body:str):
    """send out an email with the given body to all sales prospecs."""
    sg =sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email="sanojcsam123@gmail.com"
    to_email="sanojcsam26@gmail.com"
    content= Content("text/plain",body)
    mail = Mail(from_email ,to_emails ,"sales email" ,content).get()
    response = sg.client.send.post(request_body=mail)
    return {"status":"success"}

In [21]:
##agent converted  into tools
description="write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1",tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2",tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3",tool_description=description)


tools = [tool1,tool2,tool3,send_email]
tools

[FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000299091F71A0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000299091F76A0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', descr

In [14]:
##sales manager -
instruction="you are a slae manager working for abcAI. you use the tools given to you to generate cold sales email.\
    you generate email use the tools. you trll all 3sales_agent tools once before select best email"

In [ ]:
sales_manager = Agent(name="Sales Manager",instructions=instruction,tools=tools,model="")

message ="send a cold sales email addreessed to 'Dear CEO"


with trace("sales manager"):
    result = await Runner.run(sales_manager,message)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

OPENAI_API_KEY is not set, skipping trace export


In [ ]:
subject_instructions = "you can write a subject for sales email. \
    you are given amessage and need to write q subject for an email."

html_instructions= "you can convert a text email body to an html email body. \
    you are given a text email body which might have some markdown and convert it to html email body"

In [ ]:
subject_writer = Agent(name="Email subject writer" ,instructions=subject_instructions,model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer" ,tool_description="write a subject for a sale email")

html_converter = Agent(name="Html email body convertor" ,instructions=html_instructions ,model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter" ,tool_description="write atext email body to an html email body")

In [28]:
@function_tool
def send_html_email(subject:str,html_body:str) -> Dict[str,str]:

    """send out an email with the given subject and html boddy to all sales prospects"""
    sg =sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email="sanojcsam123@gmail.com"
    to_email="sanojcsam26@gmail.com"
    content= Content("text/plain",html_body)
    mail = Mail(from_email ,to_emails ,subject ,content).get()
    response = sg.client.send.post(request_body=mail)
    return {"status":"success"}

In [29]:
tools=[subject_tool,html_tool,send_email]

In [30]:
tools

[FunctionTool(name='subject_writer', description='write a subject for a sale email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000029909593F60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='write atext email body to an html email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000299095DCC20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Fun

In [32]:
instructions= "you are an email formatter and sender.you recevive the body of an email to be sent.\
    first use the subject_writer to write a subject for the email,then use the html_converter tool to convert the body to html.\
        finally,send_html_email tool use send the email with the subject and html body"

In [ ]:
emailer_agent =Agent(
    name="Email Manager",
    instructions=instructions,
    tools = tools,
    model ="gpt-4o-mini",
    handoff_description ="convert an email to html and send it"
)

In [38]:
#tools-3 handoffs-1

tools=[tool1,tool2,tool3]
handoffs=[emailer_agent]

In [42]:
sales_manager_instructions= "you are a sales manager working in abcAI company you use the tools and generate sale emails. \
    you try all 3 sales agent tools after select best one from these.\
    you select best one after picking the email,you handoff to the Email Manager agent to format and send the email"    

In [43]:
sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
) 
message = "send a sale email addressed to Dear CEO"


In [44]:
with trace("automated SDR"):
    result = await Runner.run(sales_manager,message)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

OPENAI_API_KEY is not set, skipping trace export
